In [25]:
# CELL 1: SETUP & CONFIGURATION
import os
import cv2
import pandas as pd
import numpy as np
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import tqdm

# =====================================================================
# SYSTEM CONFIGURATION PARAMETERS
# =====================================================================
BASE_PROJECT_DIR = "D:/project"
BASE_DATASET_DIR = os.path.join(BASE_PROJECT_DIR, "mcd_rppg_60_patients")
CSV_FILE_PATH = os.path.join(BASE_DATASET_DIR, "db.csv")
OUTPUT_TENSOR_DIR = os.path.join(BASE_DATASET_DIR, "cache_face_tensors_3d")
SYNC_SIGNAL_DIR = os.path.join(BASE_DATASET_DIR, "ppg_sync")

os.makedirs(OUTPUT_TENSOR_DIR, exist_ok=True)
print("✅ Cell 1 Complete: Imports loaded and directories configured.")

✅ Cell 1 Complete: Imports loaded and directories configured.


In [ ]:
# CELL 2: CORE EXTRACTION LOOP
dataset_dataframe = pd.read_csv(CSV_FILE_PATH)

# OpenCV HAAR CASCADE ROBUST LOADING
cascade_filename = "haarcascade_frontalface_default.xml"
local_cascade_path = os.path.join(BASE_PROJECT_DIR, cascade_filename)

if not os.path.exists(local_cascade_path):
    print(f"🌐 XML not found. Downloading to {local_cascade_path}...")
    url = f"https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/{cascade_filename}"
    try:
        urllib.request.urlretrieve(url, local_cascade_path)
        print("📥 Download complete!")
    except Exception as e:
        print(f"❌ Download failed: {e}. Switching to Center-Crop framework.")

face_cascade_detector = cv2.CascadeClassifier(local_cascade_path)
detector_available = not face_cascade_detector.empty()

if not detector_available:
    print("⚠️ Warning: Face detector failed to load. Defaulting to center-cropping.")
else:
    print("✅ Face detector loaded successfully!")

print("🚀 Starting production-grade 3D face tensor extraction pipeline...")

for index, row in tqdm(dataset_dataframe.iterrows(), total=len(dataset_dataframe)):
    video_filename = str(row['video']).replace('\\', '/').split('/')[-1]
    video_absolute_path = os.path.normpath(os.path.join(BASE_DATASET_DIR, 'video', video_filename))
    
    if not os.path.exists(video_absolute_path):
        continue
        
    output_filename = f"{os.path.splitext(video_filename)[0]}.npy"
    output_file_path = os.path.join(OUTPUT_TENSOR_DIR, output_filename)
    
    if os.path.exists(output_file_path):
        continue
        
    video_capture_stream = cv2.VideoCapture(video_absolute_path)
    extracted_face_sequence = []
    
    reading_success, initial_frame = video_capture_stream.read()
    if not reading_success:
        video_capture_stream.release()
        continue
        
    frame_height, frame_width, _ = initial_frame.shape
    has_detected_face = False
    
    if detector_available:
        gray_scaled_frame = cv2.cvtColor(initial_frame, cv2.COLOR_BGR2GRAY)
        detected_faces = face_cascade_detector.detectMultiScale(gray_scaled_frame, 1.3, 5)
        if len(detected_faces) > 0:
            coord_x, coord_y, width_w, height_h = detected_faces[0]
            has_detected_face = True
            
    if not has_detected_face:
        coord_x, coord_y = int(frame_width * 0.2), int(frame_height * 0.1)
        width_w, height_h = int(frame_width * 0.6), int(frame_height * 0.8)
        
    video_capture_stream.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    while video_capture_stream.isOpened():
        reading_success, standard_frame = video_capture_stream.read()
        if not reading_success: 
            break
            
        cropped_face_slice = standard_frame[coord_y:coord_y+height_h, coord_x:coord_x+width_w]
        resized_face_slice = cv2.resize(cropped_face_slice, (128, 128))
        rgb_converted_slice = cv2.cvtColor(resized_face_slice, cv2.COLOR_BGR2RGB)
        extracted_face_sequence.append(rgb_converted_slice)
        
    video_capture_stream.release()
    
    if len(extracted_face_sequence) > 0:
        numpy_tensor_matrix = np.array(extracted_face_sequence, dtype=np.uint8)
        np.save(output_file_path, numpy_tensor_matrix)

print("🎉 3D Face video spatial tensors successfully cached to storage!")

In [26]:
# CELL 3: PARTITION AND SAVE SPLITS
def partition_and_save_dataset(csv_path, save_dir, train_ratio=0.75, val_ratio=0.15, seed=42):
    df = pd.read_csv(csv_path)
    unique_patients = df['patient_id'].unique()
    
    np.random.seed(seed)
    np.random.shuffle(unique_patients)
    
    total_patients = len(unique_patients)
    train_cutoff = int(total_patients * train_ratio)
    val_cutoff = train_cutoff + int(total_patients * val_ratio)
    
    train_pids = unique_patients[:train_cutoff]
    val_pids = unique_patients[train_cutoff:val_cutoff]
    test_pids = unique_patients[val_cutoff:]
    
    train_df = df[df['patient_id'].isin(train_pids)].reset_index(drop=True)
    val_df = df[df['patient_id'].isin(val_pids)].reset_index(drop=True)
    test_df = df[df['patient_id'].isin(test_pids)].reset_index(drop=True)
    
    train_df.to_csv(os.path.join(save_dir, "production_train_split.csv"), index=False)
    val_df.to_csv(os.path.join(save_dir, "production_val_split.csv"), index=False)
    test_df.to_csv(os.path.join(save_dir, "production_test_split.csv"), index=False)
    
    print("📦 PARTITIONING METRICS & REPORT")
    print(f" • Unique Patients Count -> Train: {len(train_pids)} | Val: {len(val_pids)} | Test: {len(test_pids)}")
    print(f" • Allocated Row Indices  -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

partition_and_save_dataset(CSV_FILE_PATH, BASE_DATASET_DIR)

📦 PARTITIONING METRICS & REPORT
 • Unique Patients Count -> Train: 450 | Val: 90 | Test: 60
 • Allocated Row Indices  -> Train: 2700 | Val: 540 | Test: 360


In [27]:
# CELL 4: PYTORCH DATASET CLASS (Updated for 2D Text Files)
class ProductionFaceTensorDataset(Dataset):
    def __init__(self, csv_path, tensor_dir, sync_dir, clip_length=160):
        self.df = pd.read_csv(csv_path)
        self.tensor_dir = tensor_dir
        self.sync_dir = sync_dir
        self.clip_length = clip_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.df):
            row = self.df.iloc[idx]
            filename_base = os.path.splitext(os.path.basename(row['video']))[0]
            tensor_path = os.path.join(self.tensor_dir, f"{filename_base}.npy")
            
            if os.path.exists(tensor_path):
                break
                
            idx = (idx + 1) % len(self.df)
            attempts += 1
            
        if attempts == len(self.df):
            raise FileNotFoundError(f"❌ No valid .npy tensors found in {self.tensor_dir}")

        face_sequence = np.load(tensor_path).astype(np.float32) / 255.0  
        total_frames = face_sequence.shape[0]

        sync_path = os.path.join(self.sync_dir, f"{filename_base}.txt")
        if os.path.exists(sync_path):
            gt_signal = np.loadtxt(sync_path, dtype=np.float32)
            
            # --- THE FIX ---
            # If the text file has multiple columns (e.g. [Time, PPG]), grab only the last column
            if gt_signal.ndim > 1:
                gt_signal = gt_signal[:, -1]
        else:
            gt_signal = np.zeros(total_frames, dtype=np.float32)
            
        if len(gt_signal) > total_frames:
            gt_signal = gt_signal[:total_frames]
        else:
            gt_signal = np.pad(gt_signal, (0, max(0, total_frames - len(gt_signal))), 'edge')
            
        gt_signal = (gt_signal - np.mean(gt_signal)) / (np.std(gt_signal) + 1e-6)

        if total_frames >= self.clip_length:
            start_f = np.random.randint(0, total_frames - self.clip_length + 1)
            face_sequence = face_sequence[start_f:start_f + self.clip_length]
            gt_signal = gt_signal[start_f:start_f + self.clip_length]
        else:
            pad_len = self.clip_length - total_frames
            face_sequence = np.pad(face_sequence, ((0, pad_len), (0,0), (0,0), (0,0)), 'edge')
            gt_signal = np.pad(gt_signal, (0, pad_len), 'edge')

        video_tensor = np.transpose(face_sequence, (3, 0, 1, 2))
        true_pulse = float(row['pulse'])
        
        vitals_vector = np.array([
            row['saturation'], row['upper_ap'], row['lower_ap'],
            row['glycated_hemoglobin'], row['hemoglobin'], 
            row['cholesterol'], row['temperature']
        ], dtype=np.float32)
        
        return torch.tensor(video_tensor), torch.tensor(gt_signal), torch.tensor(true_pulse), torch.tensor(vitals_vector)

print("✅ Cell 4 Complete: Dataset Class loaded (2D TXT patch applied).")

✅ Cell 4 Complete: Dataset Class loaded (2D TXT patch applied).


In [28]:
# CELL 5: MODEL ARCHITECTURE
class SpatioTemporalFeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.conv = nn.Conv3d(hidden_dim, hidden_dim, kernel_size=3, padding=1, groups=hidden_dim)
        self.bn = nn.BatchNorm3d(hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x, T, H, W):
        B, N, D = x.shape
        x = self.fc1(x)
        x = x.view(B, T, H, W, -1).permute(0, 4, 1, 2, 3).contiguous()
        x = self.act(self.bn(self.conv(x)))
        x = x.permute(0, 2, 3, 4, 1).contiguous().view(B, N, -1)
        x = self.fc2(x)
        return x

class HybridPhysFormer(nn.Module):
    def __init__(self, num_vitals=7, dim=96):
        super().__init__()
        
        self.stem = nn.Sequential(
            nn.Conv3d(3, 24, kernel_size=(1, 5, 5), padding=(0, 2, 2)),
            nn.BatchNorm3d(24), nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)),
            
            nn.Conv3d(24, 48, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(48), nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)),
            
            nn.Conv3d(48, dim, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(dim), nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )
        
        self.tokenizer = nn.Conv3d(dim, dim, kernel_size=(4, 4, 4), stride=(4, 4, 4))
        self.ffn = SpatioTemporalFeedForward(dim=dim, hidden_dim=dim * 2)
        self.norm = nn.LayerNorm(dim)
        
        self.waveform_projector = nn.Sequential(
            nn.Linear(dim, 32), nn.GELU(), nn.Linear(32, 1)
        )
        
        self.vitals_projector = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, num_vitals)
        )

    def forward(self, x):
        B, C, T, H, W = x.shape
        x = self.stem(x) 
        
        x = self.tokenizer(x) 
        B_c, C_c, T_c, H_c, W_c = x.shape
        
        x = x.permute(0, 2, 3, 4, 1).contiguous().view(B, T_c * H_c * W_c, C_c)
        x = self.norm(x + self.ffn(x, T_c, H_c, W_c))
        
        spatial_pooled = x.view(B, T_c, H_c * W_c, C_c).mean(dim=2) 
        wave_features = spatial_pooled.permute(0, 2, 1) 
        
        wave_upsampled = F.interpolate(wave_features, size=T, mode='linear', align_corners=True)
        rppg_wave = self.waveform_projector(wave_upsampled.permute(0, 2, 1)).squeeze(-1) 
        
        global_features = spatial_pooled.mean(dim=1) 
        static_vitals = self.vitals_projector(global_features) 
        
        return rppg_wave, static_vitals

print("✅ Cell 5 Complete: Model architectures defined.")

✅ Cell 5 Complete: Model architectures defined.


In [32]:
# CELL 6: LOSS FUNCTIONS & DISTRIBUTIONS (Updated for FP16/XPU Safety)

def generate_label_distribution(target_hr, low_bpm=42, high_bpm=180, sigma=1.0):
    classes = torch.arange(low_bpm, high_bpm + 1, dtype=torch.float32, device=target_hr.device)
    target_hr = target_hr.unsqueeze(1)
    
    distribution = torch.exp(-((classes - (target_hr - 41)) ** 2) / (2 * (sigma ** 2)))
    distribution = distribution / torch.sum(distribution, dim=1, keepdim=True)
    return distribution


def calculate_ld_loss(predicted_wave, target_hr, fps=30, low_bpm=42, high_bpm=180):
    batch_size, T = predicted_wave.shape
    
    # --- THE FIX: Cast to float32 to prevent ComplexHalf FFT crashes ---
    wave_f32 = predicted_wave.to(torch.float32)
    fft_coeffs = torch.fft.rfft(wave_f32, dim=1)
    psd = torch.abs(fft_coeffs) ** 2
    
    freqs = torch.fft.rfftfreq(T, d=1.0/fps).to(predicted_wave.device)
    bpms = freqs * 60.0
    
    target_bpms = torch.arange(low_bpm, high_bpm + 1, dtype=torch.float32, device=predicted_wave.device)
    
    mapped_psd = []
    for i in range(batch_size):
        sample_psd = psd[i]
        interp_idx = torch.bucketize(target_bpms, bpms)
        clamped_indices = torch.clamp(interp_idx, 0, len(sample_psd) - 1)
        mapped_psd.append(sample_psd[clamped_indices])
        
    mapped_psd = torch.stack(mapped_psd)
    pred_distribution = F.log_softmax(mapped_psd, dim=1)
    gt_distribution = generate_label_distribution(target_hr, low_bpm, high_bpm)
    
    loss_ld = F.kl_div(pred_distribution, gt_distribution, reduction='batchmean')
    return loss_ld


def extract_hr_from_wave(pred_wave, fps=30):
    # --- THE FIX: Cast to float32 to prevent ComplexHalf FFT crashes ---
    wave_f32 = pred_wave.to(torch.float32)
    fft_data = torch.fft.rfft(wave_f32, dim=1)
    psd = torch.abs(fft_data) ** 2
    
    freqs = torch.fft.rfftfreq(pred_wave.shape[1], d=1.0/fps).to(pred_wave.device)
    bpms = freqs * 60.0
    
    valid_idx = (bpms >= 42) & (bpms <= 180)
    valid_bpms = bpms[valid_idx]
    valid_psd = psd[:, valid_idx]
    
    max_indices = torch.argmax(valid_psd, dim=1)
    estimated_hrs = valid_bpms[max_indices]
    return estimated_hrs

print("✅ Cell 6 Complete: Math and loss functions ready (FP32 safe).")

✅ Cell 6 Complete: Math and loss functions ready (FP32 safe).


In [34]:
# CELL 7: TRAINING & VALIDATION LOOP (Robust Intel XPU / PyTorch 2.0+ Precision-Safe)
import os
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

# --- NUMERICAL STABILITY UTILITIES ---
def stable_pearson_loss(pred, gt):
    """
    Computes Negative Pearson Correlation Loss.
    Casts vectors to float32 to shield operations from mixed-precision underflow (AMP).
    Uses a larger epsilon (1e-4) to prevent 0/0 division when variance is zero.
    """
    pred = pred.float()
    gt = gt.float()
    
    mean_p = torch.mean(pred, dim=1, keepdim=True)
    mean_g = torch.mean(gt, dim=1, keepdim=True)
    
    num = torch.sum((pred - mean_p) * (gt - mean_g), dim=1)
    den = torch.sqrt(torch.sum((pred - mean_p)**2, dim=1) * torch.sum((gt - mean_g)**2, dim=1) + 1e-4)
    
    # Return 1 - Pearson coefficient
    return torch.mean(1.0 - (num / den))


# --- DATASET & DATALOADER SETUP ---
train_dataset = ProductionFaceTensorDataset(
    os.path.join(BASE_DATASET_DIR, "production_train_split.csv"), OUTPUT_TENSOR_DIR, SYNC_SIGNAL_DIR, 160)
val_dataset = ProductionFaceTensorDataset(
    os.path.join(BASE_DATASET_DIR, "production_val_split.csv"), OUTPUT_TENSOR_DIR, SYNC_SIGNAL_DIR, 160)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True, pin_memory=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, pin_memory=True, num_workers=0)


# --- ROBUST DEVICE SELECTION ---
if hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch, "backends") and torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple Silicon Mac
else:
    device = torch.device("cpu")

print(f"🚀 Using device: {device}")


# --- INITIALIZE MODEL & OPTIMIZER ---
model = HybridPhysFormer(num_vitals=7).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=5e-5)


# --- HARDWARE-AGNOSTIC AMP SETUP ---
use_amp = device.type in ['cuda', 'xpu']
amp_backend = device.type if use_amp else 'cuda'
scaler = torch.amp.GradScaler(amp_backend, enabled=use_amp)

total_epochs = 25
alpha = 0.1  
best_val_loss = float('inf')

print("====================================================================")
print("🚀 BEGINNING TRAINING PIPELINE...")
print("====================================================================")

for epoch in range(1, total_epochs + 1):
    
    # ==================== TRAINING PHASE ====================
    model.train()
    running_train_total = 0.0
    actual_train_batches = 0
    beta = 1.0 * ((5.0 / 1.0) ** ((epoch - 1) / total_epochs))
    
    train_progress = tqdm(train_loader, desc=f"⏳ Epoch [{epoch}/{total_epochs}] Train", leave=False)
    
    for videos, gt_waves, target_hrs, vitals_targets in train_progress:
        if videos.shape[0] == 0 or torch.all(videos == 0): 
            continue
            
        videos, gt_waves = videos.to(device), gt_waves.to(device)
        target_hrs, vitals_targets = target_hrs.to(device), vitals_targets.to(device)
        
        optimizer.zero_grad()
        
        # Hardware-Agnostic Autocast
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            pred_wave, pred_vitals = model(videos)
            
            # HEAD A1: AMP-Safe Stable Negative Pearson Loss
            loss_time = stable_pearson_loss(pred_wave, gt_waves)
            
            # HEAD A2: Frequency Loss
            loss_frequency = calculate_ld_loss(pred_wave, target_hrs, fps=30)
            
            # HEAD B: Vitals Multi-Task Loss
            loss_vitals = F.mse_loss(pred_vitals, vitals_targets)
            
            # Balanced Compound Loss
            total_loss = (alpha * loss_time) + (beta * loss_frequency) + loss_vitals

        # --- NAN/INF EMERGENCY ESCAPE ---
        if torch.isnan(total_loss) or torch.isinf(total_loss):
            optimizer.zero_grad()
            continue # Skip batch backwards step to avoid breaking weight space

        # Scaled Backpropagation
        scaler.scale(total_loss).backward()
        
        # Exploding gradient mitigation under AMP
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        running_train_total += total_loss.item()
        actual_train_batches += 1
        train_progress.set_postfix({'Loss': f"{total_loss.item():.3f}"})
        
    # Calculate baseline train average safely
    avg_train_loss = running_train_total / actual_train_batches if actual_train_batches > 0 else float('nan')
        
        
    # ==================== VALIDATION PHASE ====================
    model.eval()
    running_val_total = 0.0
    actual_val_batches = 0
    pulse_absolute_errors, vitals_absolute_errors = [], []
    
    val_progress = tqdm(val_loader, desc=f"🔍 Epoch [{epoch}/{total_epochs}] Val", leave=False)
    
    with torch.no_grad():
        for videos, gt_waves, target_hrs, vitals_targets in val_progress:
            if videos.shape[0] == 0: 
                continue
                
            videos, gt_waves = videos.to(device), gt_waves.to(device)
            target_hrs, vitals_targets = target_hrs.to(device), vitals_targets.to(device)
            
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                pred_wave, pred_vitals = model(videos)
                
                loss_time = stable_pearson_loss(pred_wave, gt_waves)
                loss_frequency = calculate_ld_loss(pred_wave, target_hrs, fps=30)
                loss_vitals = F.mse_loss(pred_vitals, vitals_targets)
                val_loss = (alpha * loss_time) + (beta * loss_frequency) + loss_vitals
                
                # Evaluation Metrics Extraction
                estimated_hrs = extract_hr_from_wave(pred_wave, fps=30)
                pulse_absolute_errors.extend(torch.abs(estimated_hrs - target_hrs).cpu().numpy())
                vitals_absolute_errors.append(torch.mean(torch.abs(pred_vitals - vitals_targets), dim=0).cpu().numpy())
                
                running_val_total += val_loss.item()
                actual_val_batches += 1

    # --- METRICS COMPILATION ---
    avg_val_loss = running_val_total / actual_val_batches if actual_val_batches > 0 else float('nan')
    epoch_pulse_mae = np.mean(pulse_absolute_errors) if pulse_absolute_errors else float('nan')
    epoch_vitals_mae = np.mean(vitals_absolute_errors, axis=0) if vitals_absolute_errors else np.zeros(7)
    
    # Save criteria execution
    if not np.isnan(avg_val_loss) and avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), os.path.join(BASE_PROJECT_DIR, "best_production_physformer.pth"))
        save_status = "💾 (New Best Model Saved)"
    else:
        save_status = ""
    
    # --- TEXT METRICS OUTPUT ---
    print(f"\n📊 [EPOCH {epoch}/{total_epochs}] {save_status}")
    print(f" • Train Loss : {avg_train_loss:.4f} | Val Loss : {avg_val_loss:.4f}")
    print(f" • PULSE RATE MAE  : {epoch_pulse_mae:.3f} BPM")
    print(f" • MULTI-VITALS MAE: {np.mean(epoch_vitals_mae):.3f} (Overall Avg)")
    
    # Output formatting logic checking array shape bounds safely
    if len(epoch_vitals_mae) >= 7:
        print(f"   [Sat: {epoch_vitals_mae[0]:.2f} | SysBP: {epoch_vitals_mae[1]:.2f} | DiaBP: {epoch_vitals_mae[2]:.2f} | HbA1c: {epoch_vitals_mae[3]:.2f} | Hb: {epoch_vitals_mae[4]:.2f} | Chol: {epoch_vitals_mae[5]:.2f} | Temp: {epoch_vitals_mae[6]:.2f}]")    
    else:
        print(f"   [Raw Vitals Output Vector: {np.round(epoch_vitals_mae, 3)}]")
    print("-" * 70)

# Final checkpoint storage
torch.save(model.state_dict(), os.path.join(BASE_PROJECT_DIR, "final_production_physformer.pth"))
print(f"🏆 Training complete! Best Validation Loss: {best_val_loss:.4f}")

🚀 Using device: xpu
🚀 BEGINNING TRAINING PIPELINE...


RuntimeError: level_zero backend failed with error: 40 (UR_RESULT_ERROR_OUT_OF_RESOURCES)

In [33]:
# CELL 7: TRAINING & VALIDATION LOOP (Text-Only Metrics)
from tqdm import tqdm
# CELL 7: TRAINING & VALIDATION LOOP (Updated for Intel XPU / PyTorch 2.0+)
train_dataset = ProductionFaceTensorDataset(
    os.path.join(BASE_DATASET_DIR, "production_train_split.csv"), OUTPUT_TENSOR_DIR, SYNC_SIGNAL_DIR, 160)
val_dataset = ProductionFaceTensorDataset(
    os.path.join(BASE_DATASET_DIR, "production_val_split.csv"), OUTPUT_TENSOR_DIR, SYNC_SIGNAL_DIR, 160)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True, pin_memory=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, pin_memory=True, num_workers=0)

# --- ROBUST DEVICE SELECTION ---
if hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch, "backends") and torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple Silicon Mac
else:
    device = torch.device("cpu")

print(f"🚀 Using device: {device}")

model = HybridPhysFormer(num_vitals=7).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=5e-5)

# --- NEW HARDWARE-AGNOSTIC AMP SCALER ---
use_amp = device.type in ['cuda', 'xpu']
# Determine the string backend for the scaler based on the detected hardware
amp_backend = device.type if use_amp else 'cuda'
scaler = torch.amp.GradScaler(amp_backend, enabled=use_amp)

total_epochs = 25
alpha = 0.1  
best_val_loss = float('inf')

print("====================================================================")
print("🚀 BEGINNING TRAINING PIPELINE...")
print("====================================================================")

for epoch in range(1, total_epochs + 1):
    # --- TRAINING ---
    model.train()
    running_train_total = 0.0
    beta = 1.0 * ((5.0 / 1.0) ** ((epoch - 1) / total_epochs))
    
    train_progress = tqdm(train_loader, desc=f"⏳ Epoch [{epoch}/{total_epochs}] Train", leave=False)
    for videos, gt_waves, target_hrs, vitals_targets in train_progress:
        if videos.shape[0] == 0 or torch.all(videos == 0): continue
            
        videos, gt_waves = videos.to(device), gt_waves.to(device)
        target_hrs, vitals_targets = target_hrs.to(device), vitals_targets.to(device)
        
        optimizer.zero_grad()
        
        # --- NEW HARDWARE-AGNOSTIC AUTOCAST ---
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            pred_wave, pred_vitals = model(videos)
            
            # HEAD A1: Negative Pearson
            mean_p = torch.mean(pred_wave, dim=1, keepdim=True)
            mean_g = torch.mean(gt_waves, dim=1, keepdim=True)
            num = torch.sum((pred_wave - mean_p) * (gt_waves - mean_g), dim=1)
            den = torch.sqrt(torch.sum((pred_wave - mean_p)**2, dim=1) * torch.sum((gt_waves - mean_g)**2, dim=1) + 1e-6)
            loss_time = torch.mean(1.0 - (num / den))
            
            # HEAD A2: Frequency Loss
            loss_frequency = calculate_ld_loss(pred_wave, target_hrs, fps=30)
            
            # HEAD B: Vitals
            loss_vitals = F.mse_loss(pred_vitals, vitals_targets)
            total_loss = (alpha * loss_time) + (beta * loss_frequency) + loss_vitals

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_train_total += total_loss.item()
        train_progress.set_postfix({'Loss': f"{total_loss.item():.3f}"})
        
    # --- EVALUATION ---
    model.eval()
    running_val_total = 0.0
    pulse_absolute_errors, vitals_absolute_errors = [], []
    
    val_progress = tqdm(val_loader, desc=f"🔍 Epoch [{epoch}/{total_epochs}] Val", leave=False)
    with torch.no_grad():
        for videos, gt_waves, target_hrs, vitals_targets in val_progress:
            if videos.shape[0] == 0: continue
            videos, gt_waves = videos.to(device), gt_waves.to(device)
            target_hrs, vitals_targets = target_hrs.to(device), vitals_targets.to(device)
            
            # --- NEW HARDWARE-AGNOSTIC AUTOCAST ---
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                pred_wave, pred_vitals = model(videos)
                
                mean_p = torch.mean(pred_wave, dim=1, keepdim=True)
                mean_g = torch.mean(gt_waves, dim=1, keepdim=True)
                num = torch.sum((pred_wave - mean_p) * (gt_waves - mean_g), dim=1)
                den = torch.sqrt(torch.sum((pred_wave - mean_p)**2, dim=1) * torch.sum((gt_waves - mean_g)**2, dim=1) + 1e-6)
                
                loss_time = torch.mean(1.0 - (num / den))
                loss_frequency = calculate_ld_loss(pred_wave, target_hrs, fps=30)
                loss_vitals = F.mse_loss(pred_vitals, vitals_targets)
                val_loss = (alpha * loss_time) + (beta * loss_frequency) + loss_vitals
                
                estimated_hrs = extract_hr_from_wave(pred_wave, fps=30)
                pulse_absolute_errors.extend(torch.abs(estimated_hrs - target_hrs).cpu().numpy())
                vitals_absolute_errors.append(torch.mean(torch.abs(pred_vitals - vitals_targets), dim=0).cpu().numpy())
                running_val_total += val_loss.item()

    # --- METRICS COMPILATION ---
    avg_train_loss = running_train_total / len(train_loader)
    avg_val_loss = running_val_total / len(val_loader)
    epoch_pulse_mae = np.mean(pulse_absolute_errors)
    epoch_vitals_mae = np.mean(vitals_absolute_errors, axis=0)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), os.path.join(BASE_PROJECT_DIR, "best_production_physformer.pth"))
        save_status = "💾 (New Best Model Saved)"
    else:
        save_status = ""
    
    # --- TEXT METRICS OUTPUT ---
    print(f"\n📊 [EPOCH {epoch}/{total_epochs}] {save_status}")
    print(f" • Train Loss : {avg_train_loss:.4f} | Val Loss : {avg_val_loss:.4f}")
    print(f" • PULSE RATE MAE  : {epoch_pulse_mae:.3f} BPM")
    print(f" • MULTI-VITALS MAE: {np.mean(epoch_vitals_mae):.3f} (Overall Avg)")
    print(f"   [Sat: {epoch_vitals_mae[0]:.2f} | SysBP: {epoch_vitals_mae[1]:.2f} | DiaBP: {epoch_vitals_mae[2]:.2f} | HbA1c: {epoch_vitals_mae[3]:.2f} | Hb: {epoch_vitals_mae[4]:.2f} | Chol: {epoch_vitals_mae[5]:.2f} | Temp: {epoch_vitals_mae[6]:.2f}]")    
    print("-" * 70)

torch.save(model.state_dict(), os.path.join(BASE_PROJECT_DIR, "final_production_physformer.pth"))
print(f"🏆 Training complete! Best Validation Loss: {best_val_loss:.4f}")

🚀 Using device: xpu
🚀 BEGINNING TRAINING PIPELINE...



📊 [EPOCH 1/25] 
 • Train Loss : nan | Val Loss : nan
 • PULSE RATE MAE  : 57.724 BPM
 • MULTI-VITALS MAE: 16.925 (Overall Avg)
   [Sat: 43.52 | SysBP: 49.69 | DiaBP: 22.33 | HbA1c: 0.09]
----------------------------------------------------------------------



📊 [EPOCH 2/25] 
 • Train Loss : nan | Val Loss : nan
 • PULSE RATE MAE  : 54.889 BPM
 • MULTI-VITALS MAE: 2.657 (Overall Avg)
   [Sat: 2.95 | SysBP: 5.98 | DiaBP: 7.18 | HbA1c: 0.24]
----------------------------------------------------------------------



📊 [EPOCH 3/25] 
 • Train Loss : nan | Val Loss : nan
 • PULSE RATE MAE  : 54.994 BPM
 • MULTI-VITALS MAE: 2.657 (Overall Avg)
   [Sat: 2.95 | SysBP: 5.97 | DiaBP: 7.19 | HbA1c: 0.24]
----------------------------------------------------------------------


KeyboardInterrupt: 